In [ ]:
# Healthcare KPI Analysis with MONAI

## Overview

In this tutorial we demonstrate how to compute common healthcare Key Performance Indicators (KPIs) using synthetic inpatient admissions data. We illustrate how tabular health data can be integrated into analytical workflows relevant to medical AI research.

The KPIs computed in this tutorial include:

- **Average Length of Stay (LOS)**
- **30-day Readmission Rate**
- **Daily Bed Occupancy**

Although this tutorial does not use real patient data, the methodology is representative of analytics performed in hospital operations, population health management and clinical ML evaluation contexts.


## Motivation

Healthcare operations and clinical pathways generate complex tabular datasets that include admission, diagnosis and discharge patterns. These datasets complement medical imaging and can support:

- risk stratification
- resource allocation
- quality metrics
- patient flow analysis
- clinical outcome modelling

By combining synthetic EHR-like tabular data with MONAI workflows, we demonstrate how such metrics can be derived reproducibly and ethically for research and prototyping.


In [ ]:
# Install required dependencies if running standalone
# !pip install health-analytics-toolkit pandas matplotlib seaborn

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import health_analytics_toolkit as hat

sns.set(style="whitegrid")


## 1. Generate Synthetic Data

We generate a synthetic inpatient admission dataset using the `health-analytics-toolkit` package. The dataset includes:

- demographic attributes
- diagnosis codes
- admission and discharge timestamps
- hospital site codes
- readmission indicators

Synthetic data avoids patient privacy concerns while preserving realistic structure.


In [ ]:
df = hat.generate_synthetic_patients(n=2000)
df.head()


## 2. Define Cohorts

To emulate analytic workflows, we create specific patient cohorts. Cohorts can be defined by:

- age thresholds
- diagnosis categories
- hospital sites
- admission period

In real environments this supports service line analysis and operational decision-making.


In [ ]:
# Example: patients aged 65+ (elderly cohort)
elderly = hat.filter_by_age(df, min_age=65)

# Example: patients with specific chronic diagnoses
chronic_codes = ["I10", "E11", "N18"]  # hypertension, diabetes, kidney disease
chronic = hat.filter_by_diagnosis_codes(elderly, chronic_codes)

# Example: admissions to a specific hospital site
siteA = hat.filter_by_hospital_site(chronic, ["NHS-TRUST-A"])

print(f"Original dataset: {len(df)} patients")
print(f"Elderly cohort: {len(elderly)} patients")
print(f"Chronic elderly cohort: {len(chronic)} patients")
print(f"Site A chronic elderly cohort: {len(siteA)} patients")


## 3. Compute Healthcare KPIs

We compute three common hospital operations KPIs:

### **Average Length of Stay (LOS)**  
Measures inpatient duration and informs acuity, throughput and discharge planning.

### **30-day Readmission Rate**  
Proxy for care quality and care coordination, commonly monitored in public healthcare systems.

### **Daily Bed Occupancy**  
Estimates operational capacity utilisation across inpatient wards.


In [ ]:
alos = hat.average_length_of_stay(siteA)
readmit_rate = hat.readmission_rate(siteA)
bed_occ = hat.daily_bed_occupancy(siteA)

print(f"Average LOS: {alos:.2f} days")
print(f"30-day Readmission Rate: {readmit_rate:.1%}")


## 4. Visualise Outputs

Operational analytics frequently rely on visualisation to communicate patterns to clinical and administrative stakeholders.


In [ ]:
plt.figure(figsize=(12, 4))
bed_occ.plot()
plt.title("Daily Bed Occupancy (Site A Chronic Elderly Cohort)")
plt.ylabel("Occupied Beds")
plt.xlabel("Date")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
df_los = siteA.copy()
df_los["los"] = (pd.to_datetime(df_los.discharge_date) - pd.to_datetime(df_los.admission_date)).dt.days

sns.histplot(df_los["los"], bins=20, kde=False)
plt.title("Distribution of Length of Stay (days)")
plt.xlabel("Length of Stay")
plt.ylabel("Count")
plt.show()


## 5. Discussion

This workflow demonstrates that synthetic EHR-like tabular data can support healthcare analytics tasks such as:

- resource planning (bed occupancy)
- quality benchmarking (readmission)
- pathway efficiency measurement (LOS)
- cohort stratification (diagnosis, age, site)

These metrics can complement MONAI workflows that analyse medical imaging datasets, enabling multimodal clinical ML model development.

In real-world settings such analyses may contribute to:

- population health management
- service line optimisation
- discharge planning
- clinical commissioning
- digital clinical transformation programmes


## 6. Reproducibility & Notes

- This tutorial uses synthetic data to ensure full privacy compliance.
- Underlying distributions are configurable and can be adapted for benchmarking scenarios.
- No Protected Health Information (PHI) is used.
- All code is executable on standard CPUs without specialised hardware.

### Dependencies

- Python ≥ 3.9
- pandas ≥ 1.5
- health-analytics-toolkit ≥ 0.1.0
- matplotlib, seaborn (optional for plots)

### Suggested Extensions

- incorporate imaging-derived features (e.g., MONAI outputs)
- integrate survival analysis packages for clinical outcomes research
- link with FHIR-like schema for interoperability
